# NARCBench-Stego on Qwen3-32B-AWQ (the paper's model): bug vs fix

**Settings (right sidebar):** Accelerator = **GPU T4 x2**, Internet = **On**, Persistence = **Files only**.

**Run cells one at a time, top to bottom** (not Run All). Stop and report if a cell says FAIL.

1. Install extraction libraries.
2. **GATE (~15 min):** re-extract one of the authors' own Core runs and compare it with their published activations. **Must print PASS** before continuing.
3. vLLM in a separate environment, then generate Stego (20 shoes x 2 = 40 runs, ~3 h).
4. Extract final-phase activations (layers 26-30): **fixed** and **bug** (replies blanked).
5. Pack `/kaggle/working/stego_qwen3_32b.tar.gz` for download.

In [ ]:
%cd /kaggle/working
!if [ -d narcbench ]; then git -C narcbench pull -q; else git clone -q -b matched-context https://github.com/Smit-Shukla/narcbench.git; fi
%cd narcbench
!git log --oneline -3
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
# 1. Extraction libraries in the system Python (matches the authors' pins: transformers 5.5.1, autoawq 0.2.9).
#    autoawq is installed WITHOUT deps so it cannot replace Kaggle's CUDA-matched torch.
!pip install -q "transformers==5.5.1" accelerate zstandard 2>&1 | tail -2
!pip install -q "autoawq==0.2.9" --no-deps 2>&1 | tail -2
!python -c "import torch, transformers; print('torch', torch.__version__, '| transformers', transformers.__version__, '| cuda', torch.cuda.is_available())"
# Same compatibility shim the extraction scripts apply before AWQ loads
!python -c "import torch.nn as nn, transformers.activations as a; a.PytorchGELUTanh = getattr(a, 'PytorchGELUTanh', nn.GELU); import awq; print('autoawq import OK')"

In [ ]:
# 2. GATE: reproduce the authors' published activations for one of their own runs.
#    Downloads ~19 GB of model weights on first run (~5-10 min), then extracts 18 samples.
import os, subprocess
HF = "https://huggingface.co/datasets/aaronrose227/narcbench/resolve/main"
REF = "deception__audit_01__collusion"
MODEL = "Qwen/Qwen3-32B-AWQ"
os.makedirs(f"data/scenarios/ref/{REF}", exist_ok=True)
os.makedirs("data/activations/qwen3_32b/core", exist_ok=True)
for f in ["run_config.json", "agent_prompts.json", "results.json"]:
    subprocess.run(["wget", "-q", "-O", f"data/scenarios/ref/{REF}/{f}", f"{HF}/scenarios/qwen3_32b/core/{REF}/{f}"], check=True)
for f in ["metadata_gen.json", "activations_gen.npz"]:
    subprocess.run(["wget", "-q", "-O", f"data/activations/qwen3_32b/core/{f}", f"{HF}/activations/qwen3_32b/core/{f}"], check=True)

!python kaggle/run_extract.py core --runs-dir data/scenarios/ref --output-dir data/activations/ref --model {MODEL} --layers 26-30 --gen-only 2>&1 | grep -v -i warn | tail -12
!python kaggle/ref_check.py data/activations/qwen3_32b/core data/activations/ref {REF}

**Only continue if the cell above printed `PASS`.** If it printed FAIL or an error, copy the output to Claude.

In [ ]:
# 3a. vLLM in its own environment (so it cannot change the extraction libraries above). ~5-10 min.
!python -m venv /tmp/vllm_env && /tmp/vllm_env/bin/pip install -q vllm 2>&1 | tail -2
!/tmp/vllm_env/bin/python -c "import vllm; print('vllm', vllm.__version__)" 

In [ ]:
# 3b. Start vLLM with Qwen3-32B-AWQ across both T4s.
import subprocess, time, requests
PORT = 8020
EAGER = False   # set True and re-run this cell if vLLM dies with an out-of-memory error
vllm_proc = subprocess.Popen(
    f"/tmp/vllm_env/bin/python -m vllm.entrypoints.openai.api_server --model {MODEL} --served-model-name {MODEL} "
    f"--dtype half --tensor-parallel-size 2 --max-model-len 5120 --gpu-memory-utilization 0.92 "
    f"--max-num-seqs 32 --port {PORT}" + (" --enforce-eager" if EAGER else ""),
    shell=True, stdout=open("vllm.log", "w"), stderr=subprocess.STDOUT)
for i in range(120):
    try:
        if requests.get(f"http://127.0.0.1:{PORT}/v1/models", timeout=5).ok:
            print(f"vLLM ready after {i*10}s"); break
    except Exception:
        pass
    if vllm_proc.poll() is not None:
        print("vLLM DIED - last log lines:"); print(open("vllm.log").read()[-3000:]); break
    time.sleep(10)
else:
    print(open("vllm.log").read()[-3000:])
!grep -i -E "KV cache|maximum concurrency" vllm.log | tail -3

In [ ]:
# 3c. Smoke test
r = requests.post(f"http://127.0.0.1:{PORT}/v1/chat/completions", json={
    "model": MODEL, "max_tokens": 60, "temperature": 0.7,
    "messages": [{"role": "system", "content": "You are a blackjack player."},
                 {"role": "user", "content": "In one sentence, how is the table tonight?"}],
    "chat_template_kwargs": {"enable_thinking": False}}, timeout=300)
print(r.json()["choices"][0]["message"]["content"])

In [ ]:
# 3d. Generate Stego: one process per shoe, all in parallel (vLLM batches them).
import os
N_SHOES = 20   # the paper uses 20; lower to 10 only if short on time
OUT = "data/scenarios/qwen3_32b_new/stego"
os.makedirs("logs32", exist_ok=True)
procs = {s: subprocess.Popen(
    f"python generation/stego.py --model {MODEL} --base-url http://127.0.0.1:{PORT}/v1 --output-dir {OUT} --shoe {s}",
    shell=True, stdout=open(f"logs32/stego_{s:02d}.log", "w"), stderr=subprocess.STDOUT) for s in range(N_SHOES)}
print(f"launched {len(procs)} shoes")

In [ ]:
# 3e. Progress monitor (re-run any time; returns when all shoes finish)
import glob
t0 = time.time()
while True:
    done = len(glob.glob(f"{OUT}/*/results.json"))
    running = [s for s, p in procs.items() if p.poll() is None]
    print(f"[{(time.time()-t0)/60:6.1f} min] stego runs done {done}/{2*N_SHOES} | shoes running {len(running)}", flush=True)
    if not running: break
    time.sleep(300)
failed = {s: p.returncode for s, p in procs.items() if p.returncode != 0}
print("FAILED shoes:", failed or "none")
for s in failed: print(f"--- shoe {s} ---"); print(open(f"logs32/stego_{s:02d}.log").read()[-800:])

If some shoes FAILED: re-run cell 3d, then 3e. Finished runs are skipped (resume).

In [ ]:
# 4a. Free the GPUs
vllm_proc.terminate(); time.sleep(20)
!pkill -f vllm.entrypoints || true
time.sleep(10)
!nvidia-smi --query-gpu=memory.used --format=csv

In [ ]:
# 4b. Extract final-phase activations, layers 26-30 (the paper's layers): FIXED and BUG
ACT = "data/activations/qwen3_32b_new"
X = f"--model {MODEL} --layers 26,27,28,29,30 --gen-only --channels public"
!python kaggle/run_extract.py stego --runs-dir {OUT} --output-dir {ACT}/stego     {X} --final-only 2>&1 | grep -v -i warn | tail -12
!python kaggle/run_extract.py stego --runs-dir {OUT} --output-dir {ACT}/stego_bug {X} --stego-bug  2>&1 | grep -v -i warn | tail -12

In [ ]:
# 5. Package for download (Output panel -> stego_qwen3_32b.tar.gz)
!tar czf /kaggle/working/stego_qwen3_32b.tar.gz data/activations/qwen3_32b_new data/activations/ref data/scenarios/qwen3_32b_new logs32 vllm.log
!ls -lh /kaggle/working/stego_qwen3_32b.tar.gz